## Business Context
As electric vehicles become more common, the demand for charging stations is growing. Operators ofter don't know when and where demand will be high, which leads to queues at busy chargers while other chargers sit idle. This can frustrate drivers and cost operators customers. 

This project uses about 72000 charging sessions recorded between september 2021 and september 2022. Each session includes the user, charger, charger company, location, charger type, start and end time, duration and the energy used (Demand). I analyse these usage patterns and build models to predict and forecast charging demand.

## Business Objectives
1. Predict session demand
2. Classify high-demand sessions
3. Forecast future demand
4. Explain Driving Factors
5. Support Business Decisions

## Analysis Questions
1. How is charging demand distributed? Is it skewed, and are there any extremes?
2. How long do charging sessions usually last?
3. Which locations, charger types and charger companies have the most sessions?
4. Which locations have the highest average demand per session?
5. Does demand differ between charger types (0 vs 1) and charger companies (0 vs 1)?
6. Which hours of the day and days of the week have the most sessions and the highest demand?
7. How strongly is session duration related to demand?
8. Do weekdays and weekends show different charging behaviour?
9. How does total daily and monthly demand change over the year? Is there a trend or a weekly pattern?
10. Are some users and chargers used much more than others? Are there underused chargers?
11. How does demand vary by location and hour together? For example do hotels peak at night and companies during the day?
12. Does the effect of charger type on demand change across locations?
13. Which variables are most correlated with demand?

## Data Dictionary

In [1]:
# Creating the data dictionary with all the columns, their data types, description of each columns and their roles.
import pandas as pd

data_dict = pd.DataFrame({
    'Column': [
        'UserID', 'ChargerID', 'ChargerCompany', 'Location', 'ChargerType', 'StartDay', 
        'StartTime', 'EndDay', 'EndTime', 'StartDatetime', 'EndDatetime', 'Duration', 'Demand'
    ],
    'Raw dtype': [
        'int64', 'int64', 'int64', 'object', 'int64', 'object', 
        'object', 'object', 'object', 'object', 'object', 'int64', 'float64'
    ],
    'Final dtype': [
        'int64', 'int64', 'int64', 'object', 'int64', 'dropped', 
        'dropped', 'dropped', 'dropped','datetime64', 'datetime64', 'int64', 'float64'
    ],
    'Description': [
        'Anonymised ID of the user',
        'Anonymised ID of the charger',
        'Company operating the charger (encoded 0/1)',
        'Type of place where the charger is installed (14 categories, e.g. hotel, apartment)',
        'Type of charger (encoded 0/1)',
        'Date the charging session started',
        'Time the charging session started',
        'Date the charging session ended',
        'Time the charging session ended',
        'Date and time the charging session started',
        'Date and time the charging session ended',
        'Length of the charging session in minutes',
        'Energy used in the charging session (likely kWh)'
    ],
    'Role': [
        'ID - used to create historical features',
        'ID - used to create historical features',
        'Feature',
        'Feature',
        'Feature',
        'Redundant - same as StartDatetime',
        'Redundant - same as StartDatetime',
        'Redundant - same as EndDatetime',
        'Redundant - same as EndDatetime',
        'Time - source of hour/day/month features and time-series index',
        'Leakage - known only after the session ends',
        'Leakage - known only after the session ends',
        'Target'
    ]
})

data_dict.to_excel('../docs/data_dictionary.xlsx', index=False)
pd.read_excel('../docs/data_dictionary.xlsx')

,Column,Raw dtype,Final dtype,Description,Role
0,UserID,int64,int64,Anonymised ID of the user,ID - used to create historical features
1,ChargerID,int64,int64,Anonymised ID of the charger,ID - used to create historical features
2,ChargerCompany,int64,int64,Company operating the charger (encoded 0/1),Feature
3,Location,object,object,Type of place where the charger is installed (...,Feature
4,ChargerType,int64,int64,Type of charger (encoded 0/1),Feature
5,StartDay,object,dropped,Date the charging session started,Redundant - same as StartDatetime
6,StartTime,object,dropped,Time the charging session started,Redundant - same as StartDatetime
7,EndDay,object,dropped,Date the charging session ended,Redundant - same as EndDatetime
8,EndTime,object,dropped,Time the charging session ended,Redundant - same as EndDatetime
9,StartDatetime,object,datetime64,Date and time the charging session started,Time - source of hour/day/month features and t...


In [2]:
# Loading the Raw Data
df = pd.read_csv('../data/raw/ChargingRecords.csv')
df

,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand
0,0,1,1,hotel,0,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54,2022-09-15 23:59,185,20.36
1,0,1,1,hotel,0,2022-09-14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01,2022-09-14 21:31,90,10.19
2,0,1,1,hotel,0,2022-09-14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54,2022-09-14 19:54,60,6.78
3,0,1,1,hotel,0,2022-09-29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32,2022-09-30 0:16,344,37.65
4,0,1,1,hotel,0,2022-09-25,19:30:15,2022-09-26,0:30:14,2022-09-25 19:30,2022-09-26 0:30,300,33.81
...,...,...,...,...,...,...,...,...,...,...,...,...,...
72851,2155,2649,0,public institution,0,2022-01-27,9:54:44,2022-01-27,9:58:57,2022-01-27 9:54,2022-01-27 9:58,4,0.50
72852,2379,2670,0,sightseeing,1,2021-10-31,14:52:11,2021-10-31,15:20:28,2021-10-31 14:52,2021-10-31 15:20,28,16.53
72853,2388,2670,0,sightseeing,1,2021-10-03,13:52:14,2021-10-03,14:32:13,2021-10-03 13:52,2021-10-03 14:32,40,12.20
72854,2388,2671,0,company,1,2021-11-18,11:37:44,2021-11-18,11:45:37,2021-11-18 11:37,2021-11-18 11:45,8,3.80


In [3]:
# checking the shape
df.shape

(72856, 13)

In [5]:
# Checking first few rows
df.head()

,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand
0,0,1,1,hotel,0,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54,2022-09-15 23:59,185,20.36
1,0,1,1,hotel,0,2022-09-14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01,2022-09-14 21:31,90,10.19
2,0,1,1,hotel,0,2022-09-14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54,2022-09-14 19:54,60,6.78
3,0,1,1,hotel,0,2022-09-29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32,2022-09-30 0:16,344,37.65
4,0,1,1,hotel,0,2022-09-25,19:30:15,2022-09-26,0:30:14,2022-09-25 19:30,2022-09-26 0:30,300,33.81


In [7]:
# checking the info of the columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72856 entries, 0 to 72855
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   UserID          72856 non-null  int64  
 1   ChargerID       72856 non-null  int64  
 2   ChargerCompany  72856 non-null  int64  
 3   Location        72856 non-null  object 
 4   ChargerType     72856 non-null  int64  
 5   StartDay        72856 non-null  object 
 6   StartTime       72856 non-null  object 
 7   EndDay          72856 non-null  object 
 8   EndTime         72856 non-null  object 
 9   StartDatetime   72856 non-null  object 
 10  EndDatetime     72856 non-null  object 
 11  Duration        72856 non-null  int64  
 12  Demand          72856 non-null  float64
dtypes: float64(1), int64(5), object(7)
memory usage: 7.2+ MB


In [ ]:
# Checking statistical values of all the columns 
df.describe()

,UserID,ChargerID,ChargerCompany,ChargerType,Duration,Demand
count,72856.000000,72856.000000,72856.000000,72856.00000,72856.000000,72856.000000
mean,338.164667,525.489212,0.627443,0.20524,151.194054,17.444445
std,553.970996,569.544469,0.483489,0.40388,146.430286,13.492728
min,0.000000,1.000000,0.000000,0.00000,-29.000000,0.010000
25%,0.000000,107.000000,0.000000,0.00000,40.000000,7.530000
50%,62.000000,301.000000,1.000000,0.00000,112.000000,14.100000
75%,407.000000,738.000000,1.000000,0.00000,202.000000,23.200000
max,2501.000000,2671.000000,1.000000,1.00000,1573.000000,97.000000


In [11]:
# Checking the unique value counts of all the columns
df.nunique()

UserID             2337
ChargerID          2119
ChargerCompany        2
Location             14
ChargerType           2
StartDay            366
StartTime         43245
EndDay              365
EndTime           46710
StartDatetime     65628
EndDatetime       66745
Duration            979
Demand             6046
dtype: int64

In [12]:
# Checking value counts of categorical columns
df['Location'].value_counts()

Location
public area           14082
apartment             14038
resort                 8854
hotel                  8600
company                7558
public institution     7023
public parking lot     3396
market                 3005
restaurant             2222
camping                1630
sightseeing            1203
golf                   1029
accommodation           187
bus garage               29
Name: count, dtype: int64

In [14]:
df['ChargerType'].value_counts()

ChargerType
0    57903
1    14953
Name: count, dtype: int64

In [15]:
df['ChargerCompany'].value_counts()

ChargerCompany
1    45713
0    27143
Name: count, dtype: int64

In [16]:
# Checking the date range of the data
df['StartDatetime'].min()

'2021-09-30 17:14'

In [17]:
df['StartDatetime'].max()

'2022-09-30 9:52'

#### Observations
1. The data has 72,856 charging sessions and 13 columns. It covers one year, from Sep 2021 to Sep 2022.
2. I didn’t find any nulls in info(). I’ll check this again properly in the data quality notebook.
3. All the date and time columns are stored as object (text), so I need to convert them to datetime.
4. Duration has negative values (min is −29). 
Most likely the reason could be the end time is before the start time which look like recording errors. 
I’ll confirm and handle them in cleaning.
5. ChargerType and ChargerCompany have only two values each (0 and 1). The dataset doesn’t say what they actually mean.
6. There are 14 different locations. Public area and apartment have the most sessions, and bus garage has only 29.
7. Demand is right skewed. The mean (17.4) is higher than the median (14.1), so a few sessions use a lot more energy than normal.